<a href="https://colab.research.google.com/github/Varanapat/SUPERAI_SS6/blob/main/605558_Mini_Hackathon_1_Online.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMPORT LIBRARIES

นำเข้าไลบรารีทั้งหมดที่จะใช้ตลอด pipeline โดยครอบคลุมทุกขั้นตอนตั้งแต่การจัดการข้อมูลไปจนถึงการวิเคราะห์ผลลัพธ์

**ไลบรารีหลักที่ใช้:**
- `numpy`, `pandas` — จัดการข้อมูลตัวเลขและตาราง
- `matplotlib`, `seaborn` — สร้างกราฟและ visualization
- `sklearn` — ชุดเครื่องมือ Machine Learning ครบวงจร (preprocessing, model selection, metrics)
- `lightgbm` — โมเดล Gradient Boosting ประสิทธิภาพสูง เหมาะกับข้อมูลขนาดใหญ่
- `shap` — อธิบายผลลัพธ์ของโมเดล (Explainability)


การใช้ `warnings.filterwarnings("ignore")` ช่วยซ่อน warning ที่ไม่จำเป็น ทำให้ output อ่านง่ายขึ้น และการตั้ง `matplotlib.use("Agg")` ช่วยให้บันทึกกราฟลงไฟล์ได้โดยไม่ต้องแสดงหน้าต่าง GUI

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, roc_curve)
from sklearn.calibration import CalibratedClassifierCV
import lightgbm as lgb
import shap
import os

กำหนดค่าคงที่ที่ใช้ตลอด pipeline

- `SEED = 42` — กำหนด Random Seed เพื่อให้ผลลัพธ์ทำซ้ำได้ทุกครั้งที่รัน (Reproducibility)
- `np.random.seed(SEED)` — ล็อก seed ของ NumPy
- `OUT_DIR = '/output'` — กำหนด path สำหรับบันทึกไฟล์ผลลัพธ์ เช่น กราฟและไฟล์ submission

การตั้งค่าเหล่านี้เป็นขั้นตอนมาตรฐานในการทำ Machine Learning เพื่อให้ผลการทดลองมีความสม่ำเสมอและตรวจสอบได้

In [ ]:
SEED = 42
np.random.seed(SEED)
OUT_DIR = '/output/picture'

#  1. LOAD DATA

หลดไฟล์ข้อมูลทั้งหมดที่จำเป็นสำหรับ pipeline

- `train.csv` — ชุดข้อมูล Training ที่มี label `clicked`
- `test.csv` — ชุดข้อมูล Test สำหรับทำนาย (ไม่มี label)
- `sample_submission.csv` — รูปแบบไฟล์ submission ที่ต้องส่ง

หลังโหลดข้อมูล จะพิมพ์สรุปขนาดของ DataFrame และ **Click Rate** ของ training set เพื่อให้เห็น class imbalance ตั้งแต่ต้น ซึ่งเป็นข้อมูลสำคัญในการเลือก metric และ strategy การ train โมเดลต่อไป

**บทบาทใน Pipeline:** จุดเริ่มต้นของ pipeline ทั้งหมด ข้อมูลที่โหลดในขั้นตอนนี้จะถูกนำไปใช้ในทุกขั้นตอนถัดไป

In [ ]:
train = pd.read_csv(f"train.csv")
test  = pd.read_csv(f"test.csv")
sub   = pd.read_csv(f"sample_submission.csv")

In [ ]:
print(f"Train : {train.shape[0]:,} rows × {train.shape[1]} cols")
print(f"Test  : {test.shape[0]:,} rows × {test.shape[1]} cols")
print(f"Target click-rate: {train['clicked'].mean():.4f}  "
      f"({train['clicked'].sum():,} clicks / {len(train):,} impressions)")

Train : 374,590 rows × 28 cols
Test  : 125,410 rows × 27 cols
Target click-rate: 0.3069  (114,957 clicks / 374,590 impressions)


#  2. EDA  — key charts saved to disk

สร้างกราฟ EDA ชุดแรก เพื่อทำความเข้าใจลักษณะของข้อมูลก่อนสร้างโมเดล ประกอบด้วย 3 กราฟในภาพเดียวกัน (`eda_overview.png`):

1. **Class Distribution** — แสดงสัดส่วน clicked vs. not-clicked เพื่อประเมิน class imbalance
2. **CTR by Day of Week** — แสดงอัตราการคลิกแยกตามวันในสัปดาห์ ช่วยระบุวันที่ผู้ใช้งาน engage สูงสุด
3. **CTR by Hour of Day** — แสดง pattern การคลิกตามช่วงเวลา ช่วยระบุ peak hours

**บทบาทใน Pipeline:** EDA ช่วยให้ตัดสินใจได้ว่าควร engineer feature ใดบ้าง เช่น การรู้ว่า CTR แตกต่างกันตามเวลา นำไปสู่การสร้าง `is_peak_hour` ในขั้นตอน Feature Engineering

In [ ]:
## 2a. Target distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("EDA — Click-Through Rate Overview", fontsize=14, fontweight="bold")


Text(0.5, 0.98, 'EDA — Click-Through Rate Overview')

In [ ]:
# Class balance
vals = train["clicked"].value_counts()
axes[0].bar(["Not Clicked", "Clicked"], vals.values, color=["#4C72B0", "#DD8452"])
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(vals.values):
    axes[0].text(i, v + 2000, f"{v:,}\n({v/len(train)*100:.1f}%)", ha="center")


In [ ]:
# CTR by day of week
ctr_dow = train.groupby("day_of_week")["clicked"].mean().sort_values(ascending=False)
axes[1].bar(ctr_dow.index, ctr_dow.values, color="#55A868")
axes[1].set_title("CTR by Day of Week")
axes[1].set_ylabel("Click Rate")
axes[1].tick_params(axis="x", rotation=45)

In [ ]:
# CTR by hour
ctr_hour = train.groupby("hour")["clicked"].mean()
axes[2].plot(ctr_hour.index, ctr_hour.values, marker="o", color="#C44E52")
axes[2].set_title("CTR by Hour of Day")
axes[2].set_xlabel("Hour")
axes[2].set_ylabel("Click Rate")
axes[2].grid(alpha=0.3)

plt.tight_layout()
!mkdir -p /output/picture
plt.savefig(f"{OUT_DIR}/eda_overview.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✓ Saved eda_overview.png")


  ✓ Saved eda_overview.png


สร้างกราฟ bar chart จำนวน 6 กราฟ (`eda_categorical.png`) เพื่อแสดง Click Rate แยกตามฟีเจอร์เชิงหมวดหมู่ที่สำคัญ ได้แก่:

- `device_type` — ประเภทอุปกรณ์
- `device_conn_type` — ประเภทการเชื่อมต่ออินเทอร์เน็ต
- `creative_size` — ขนาดของโฆษณา
- `user_segment` — กลุ่มผู้ใช้
- `site_category` — ประเภทเว็บไซต์
- `banner_pos` — ตำแหน่งแบนเนอร์

การวิเคราะห์นี้ช่วยระบุว่าฟีเจอร์ใดมี discriminative power สูง ซึ่งนำไปสู่การ encode อย่างเหมาะสมในขั้นตอน preprocessing

In [ ]:
## 2b. Feature-level CTR charts
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("CTR by Key Categorical Features", fontsize=14, fontweight="bold")

cat_pairs = [
    ("device_type",    axes[0][0]),
    ("device_conn_type", axes[0][1]),
    ("creative_size",  axes[0][2]),
    ("user_segment",   axes[1][0]),
    ("site_category",  axes[1][1]),
    ("banner_pos",     axes[1][2]),
]
for col, ax in cat_pairs:
    ctr = train.groupby(col)["clicked"].mean().sort_values(ascending=False)
    ax.bar(ctr.index.astype(str), ctr.values, color="#8172B2")
    ax.set_title(f"CTR by {col}")
    ax.set_ylabel("Click Rate")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/eda_categorical.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✓ Saved eda_categorical.png")

  ✓ Saved eda_categorical.png


In [ ]:
# !ls /output/picture

สร้าง histogram เปรียบเทียบการกระจายตัวของ numerical features ระหว่างกลุ่ม `clicked=True` (สีน้ำเงิน) และ `clicked=False` (สีส้ม) สำหรับ 8 features หลัก:

`ad_quality_score`, `user_depth`, `historical_user_ctr`, `C1`, `C15`, `C16`, `C21`, `banner_pos`

**จุดสำคัญ:** หาก distribution ของทั้งสองกลุ่มต่างกันชัดเจน แสดงว่า feature นั้น predict การคลิกได้ดี ผลจาก EDA นี้ใช้ยืนยันว่าควรนำ feature ใดเข้าโมเดล

In [ ]:
## 2c. Numerical distributions
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Numeric Feature Distributions (clicked=blue / not=orange)", fontsize=13, fontweight="bold")
num_cols = ["ad_quality_score", "user_depth", "historical_user_ctr",
            "C1", "C15", "C16", "C21", "banner_pos"]
for i, col in enumerate(num_cols):
    ax = axes[i // 4][i % 4]
    for label, color in [(True, "#4C72B0"), (False, "#DD8452")]:
        ax.hist(train.loc[train["clicked"] == label, col].dropna(),
                bins=30, alpha=0.5, color=color, label=str(label), density=True)
    ax.set_title(col)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/eda_numerics.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✓ Saved eda_numerics.png")

  ✓ Saved eda_numerics.png


สร้าง heatmap แสดงความสัมพันธ์เชิงเส้น (Pearson correlation) ระหว่าง numerical features และ target variable `clicked`

**ข้อมูลที่ได้:**
- `historical_user_ctr` มีความสัมพันธ์สูงสุดกับ `clicked` — บ่งชี้ว่าพฤติกรรมการคลิกในอดีตเป็น predictor ที่ดีที่สุด
- `ad_quality_score` มีความสัมพันธ์กับ target ในระดับปานกลาง
- Features ที่มี correlation ต่ำกับ target แต่สูงกับ feature อื่น อาจเกิด multicollinearity ได้

ข้อมูลจากกราฟนี้ช่วยวางแผน feature engineering ในขั้นตอนถัดไป

In [ ]:
## 2d. Correlation heatmap
num_train = train[num_cols + ["clicked"]].copy()
num_train["clicked"] = num_train["clicked"].astype(int)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(num_train.corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix (numeric features + target)", fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/eda_correlation.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✓ Saved eda_correlation.png")

print("\n  KEY INSIGHTS:")
print(f"  • historical_user_ctr corr with clicked: "
      f"{num_train.corr()['clicked']['historical_user_ctr']:.3f}")
print(f"  • ad_quality_score corr with clicked   : "
      f"{num_train.corr()['clicked']['ad_quality_score']:.3f}")
print(f"  • CTR is highest on: {ctr_dow.index[0]}")

  ✓ Saved eda_correlation.png

  KEY INSIGHTS:
  • historical_user_ctr corr with clicked: 0.149
  • ad_quality_score corr with clicked   : -0.005
  • CTR is highest on: Sun


#  3. PREPROCESSING & FEATURE ENGINEERING


รวม `train` และ `test` DataFrame เข้าด้วยกันเป็น `full` เพื่อให้การ encode ฟีเจอร์มีความสม่ำเสมอระหว่างทั้งสองชุด

- เพิ่ม column `_split` เพื่อแยก train/test ได้ในภายหลัง
- เติม `clicked = NaN` ให้กับ test set (ยังไม่มี label)
- ใช้ `pd.concat()` รวมข้อมูล

**เหตุผลสำคัญ:** หากทำ encoding แยกกัน อาจทำให้ category ในชุด test บางตัวถูก encode เป็นค่าต่างจากชุด train ส่งผลให้โมเดลทำงานผิดพลาดได้

In [ ]:
## 3a. Combine for consistent encoding
train["_split"] = "train"
test["_split"]  = "test"
test["clicked"] = np.nan
full = pd.concat([train, test], ignore_index=True, sort=False)

จัดการ missing values ใน columns ที่เกี่ยวกับ site และ app

```python
for col in ["site_id", "site_domain", "app_id", "app_domain"]:
    full[col] = full[col].fillna("NONE")
```

**เหตุผล:** เมื่อ `is_app=True` โฆษณาจะแสดงบน mobile app ดังนั้น `site_id` และ `site_domain` จะเป็น NaN ตามธรรมชาติ และในทางกลับกัน การเติม `"NONE"` แทนการ drop ช่วยรักษาข้อมูลและให้โมเดลเรียนรู้ว่า "ไม่มีข้อมูล" ก็เป็น pattern หนึ่ง

In [ ]:
## 3b. Handle missing values in site/app columns
# When is_app=True → site cols are N/A and vice versa; fill with "NONE"
for col in ["site_id", "site_domain", "app_id", "app_domain"]:
    full[col] = full[col].fillna("NONE")

สร้างฟีเจอร์ใหม่จาก `timestamp` และ `day_of_week` เพื่อให้โมเดลเข้าใจ temporal patterns

- `ts_hour`, `ts_minute` — ชั่วโมงและนาทีจาก timestamp จริง
- `is_weekend` — flag ว่าเป็นวันหยุดสุดสัปดาห์หรือไม่ (0/1)
- `is_peak_hour` — flag ว่าอยู่ในช่วง Peak Hours (9–12 น. หรือ 17–21 น.) หรือไม่

**บทบาทใน Pipeline:** ฟีเจอร์เหล่านี้ช่วยให้โมเดลเรียนรู้ว่าช่วงเวลาใดผู้ใช้มีแนวโน้มคลิกโฆษณาสูงกว่า ซึ่งสอดคล้องกับผลจาก EDA

In [ ]:
## 3c. Feature engineering
print("  Engineering features …")

  Engineering features …


In [ ]:
# Time-based features
full["timestamp"] = pd.to_datetime(full["timestamp"])
full["ts_hour"]   = full["timestamp"].dt.hour
full["ts_minute"] = full["timestamp"].dt.minute
full["is_weekend"]= full["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)
full["is_peak_hour"] = full["hour"].apply(lambda h: 1 if 9<=h<=12 or 17<=h<=21 else 0)


แปลง `creative_size` (เช่น `"300x250"`) ให้เป็นตัวเลขที่โมเดลใช้ได้

```python
def parse_size(s):
    w, h = s.split("x")
    return int(w), int(h)
```

ฟีเจอร์ที่สร้างใหม่:
- `banner_width`, `banner_height` — ความกว้างและความสูงของโฆษณา
- `banner_area` — พื้นที่รวม (width × height) สะท้อนความโดดเด่นของโฆษณา
- `banner_aspect` — อัตราส่วนภาพ (width / height) สะท้อนรูปทรงของโฆษณา

ข้อมูลเชิงขนาดเหล่านี้มักมีความสัมพันธ์กับ CTR เนื่องจากโฆษณาขนาดใหญ่มักเตะตาผู้ใช้มากกว่า

In [ ]:
# Creative size → width & height
def parse_size(s):
    try:
        w, h = s.split("x")
        return int(w), int(h)
    except:
        return np.nan, np.nan

full[["banner_width", "banner_height"]] = full["creative_size"].apply(
    lambda s: pd.Series(parse_size(s)))
full["banner_area"] = full["banner_width"] * full["banner_height"]
full["banner_aspect"] = (full["banner_width"] / (full["banner_height"] + 1e-6)).round(4)


สร้างฟีเจอร์เชิงพฤติกรรมผู้ใช้และใช้เทคนิค **Target Encoding** ซึ่งเป็นหัวใจสำคัญของ pipeline นี้

**ฟีเจอร์ User Engagement:**
- `is_high_depth` — flag ว่า user มี engagement ลึก (user_depth ≥ 5)
- `ctr_x_quality` — interaction feature ระหว่าง CTR ในอดีตกับคะแนนคุณภาพโฆษณา

**Target Encoding (Bayesian Smoothing):**
```python
def target_encode(col, smoothing=20):
    # ป้องกัน overfitting ด้วย Bayesian smoothing
    stats["te"] = (stats["sum"] + smoothing * global_mean) / (stats["count"] + smoothing)
```

เทคนิคนี้แปลง high-cardinality categorical columns (เช่น `publisher_id`, `ad_id`) ให้เป็นตัวเลขโดยใช้ **ค่าเฉลี่ยของ target จากชุด train เท่านั้น** (ป้องกัน data leakage) และใช้ Bayesian smoothing เพื่อลด noise จาก categories ที่มีข้อมูลน้อย

In [ ]:
# User engagement features
full["is_high_depth"] = (full["user_depth"] >= 5).astype(int)
full["ctr_x_quality"] = full["historical_user_ctr"] * full["ad_quality_score"]
full["quality_bucket"] = pd.qcut(full["ad_quality_score"], 5,
                                  labels=False, duplicates="drop")


In [ ]:
# Target encoding helpers (computed from TRAIN only)
tr_mask = full["_split"] == "train"
clicked_num = full.loc[tr_mask, "clicked"].map({True: 1, False: 0, 1: 1, 0: 0})

def target_encode(col, smoothing=20):
    """Bayesian-smoothed target encoding (to avoid leakage)."""
    global_mean = clicked_num.mean()
    stats = (full.loc[tr_mask, col]
                 .to_frame()
                 .assign(y=clicked_num.values)
                 .groupby(col)["y"]
                 .agg(["sum", "count"]))
    stats["te"] = (stats["sum"] + smoothing * global_mean) / (stats["count"] + smoothing)
    return full[col].map(stats["te"]).fillna(global_mean)

for col in ["publisher_id", "ad_id", "ad_campaign_id", "device_model",
            "site_id", "site_domain", "app_id", "app_domain"]:
    full[f"te_{col}"] = target_encode(col)
    print(f"    ✓ Target-encoded {col}")


    ✓ Target-encoded publisher_id
    ✓ Target-encoded ad_id
    ✓ Target-encoded ad_campaign_id
    ✓ Target-encoded device_model
    ✓ Target-encoded site_id
    ✓ Target-encoded site_domain
    ✓ Target-encoded app_id
    ✓ Target-encoded app_domain


คำนวณ aggregated statistics ของแต่ละ publisher จากชุด train แล้ว merge กลับเข้าสู่ dataset หลัก

- `pub_impression_cnt` — จำนวน impression ทั้งหมดของ publisher นั้น (สะท้อนขนาด/ความนิยม)
- `pub_ctr` — อัตราการคลิกเฉลี่ยของ publisher นั้น (สะท้อนคุณภาพ inventory)

**บทบาทใน Pipeline:** ฟีเจอร์เหล่านี้ให้ข้อมูลระดับ publisher ซึ่งเป็น group-level signal ที่สำคัญ เนื่องจาก publisher แต่ละรายมีคุณภาพ traffic แตกต่างกันอย่างมีนัยสำคัญ

In [ ]:
# Publisher-level stats
pub_stats = (full.loc[tr_mask]
               .groupby("publisher_id")
               .agg(pub_impression_cnt=("clicked", "count"),
                    pub_ctr=("clicked", lambda x: x.map({True:1,False:0}).mean()))
               .reset_index())
full = full.merge(pub_stats, on="publisher_id", how="left")
full["pub_impression_cnt"] = full["pub_impression_cnt"].fillna(0)
full["pub_ctr"] = full["pub_ctr"].fillna(full.loc[tr_mask, "pub_ctr"].mean())


แปลง categorical features ที่เหลือให้เป็นตัวเลข โดยใช้ `LabelEncoder` จาก scikit-learn

Columns ที่ encode: `day_of_week`, `device_type`, `device_conn_type`, `site_category`, `user_segment`, `creative_size`

`LabelEncoder` จะแมปแต่ละ category เป็นตัวเลข integer (เช่น "Monday" → 0, "Tuesday" → 1) โมเดลอย่าง LightGBM สามารถจัดการ ordinal encoding แบบนี้ได้ดี โดยเก็บ encoder ไว้ใน `le_dict` เผื่อใช้ decode ผลลัพธ์ในภายหลัง

In [ ]:
## 3d. Encode remaining categoricals
le_cols = ["day_of_week", "device_type", "device_conn_type", "site_category",
           "user_segment", "creative_size"]
le_dict = {}
for col in le_cols:
    le = LabelEncoder()
    full[col] = le.fit_transform(full[col].astype(str))
    le_dict[col] = le

full["is_app"] = full["is_app"].astype(int)


In [ ]:
## 3e. Drop non-feature columns
DROP = ["impression_id", "user_id", "timestamp", "device_model",
        "site_id", "site_domain", "app_id", "app_domain",
        "ad_id", "ad_campaign_id", "publisher_id", "_split"]

FEATURES = [c for c in full.columns if c not in DROP + ["clicked"]]
print(f"\n  Total features: {len(FEATURES)}")




  Total features: 37


ทำสองสิ่งสำคัญ:

1. **กำหนด Feature List:** ลบ columns ที่ไม่ใช่ feature จริง (เช่น `impression_id`, `user_id`, `timestamp`) ออก รวมถึง raw ID columns ที่ถูกแทนด้วย target-encoded versions แล้ว

2. **แบ่งข้อมูลกลับ:** แยก `full` DataFrame กลับเป็น `train_fe` และ `test_fe` พร้อมสร้าง:
   - `X` — feature matrix ของ train
   - `y` — target vector (แปลงเป็น 0/1)
   - `X_test` — feature matrix ของ test

**บทบาทใน Pipeline:** นี่คือขั้นตอนสุดท้ายของ preprocessing ที่เตรียมข้อมูลให้พร้อมสำหรับการ train โมเดลในขั้นตอนถัดไป


In [ ]:
## 3f. Split back
train_fe = full[full["_split_orig"] if "_split_orig" in full.columns else (full["_split"] == "train" if "_split" in full.columns else full.index < len(train))]


In [ ]:
# Re-split properly
full["_split"] = full["_split"].fillna("test")
train_fe = full[full["_split"] == "train"].copy()
test_fe  = full[full["_split"] == "test"].copy()

y = train_fe["clicked"].map({True: 1, False: 0}).astype(int)
X = train_fe[FEATURES]
X_test = test_fe[FEATURES]

print(f"  X_train: {X.shape},  X_test: {X_test.shape}")
print(f"  Target balance: {y.mean():.4f} positive rate")


  X_train: (374590, 37),  X_test: (125410, 37)
  Target balance: 0.3069 positive rate


#  4. MODELING

สร้างโมเดล **Baseline** ด้วย Logistic Regression โดยใช้ `Pipeline` ของ scikit-learn ที่ประกอบด้วย:

1. `StandardScaler` — ปรับ scale ของ features (จำเป็นสำหรับ Logistic Regression)
2. `LogisticRegression` — โมเดลเชิงเส้นพื้นฐาน (C=0.1 เพื่อ regularization)

ประเมินผลด้วย **5-Fold Stratified Cross-Validation** และ metric **ROC-AUC** เพื่อให้รับมือกับ class imbalance ได้ดี จากนั้น train บนข้อมูลทั้งหมดและทำนาย test set เก็บไว้สำหรับ ensemble ในภายหลัง

**บทบาทใน Pipeline:** Logistic Regression ทำหน้าที่เป็น baseline ที่ให้เปรียบเทียบว่าโมเดลที่ซับซ้อนกว่าทำได้ดีขึ้นอีกมากน้อยแค่ไหน

In [ ]:
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)



  STEP 4: MODELING


In [ ]:
# ── 4a. Logistic Regression (baseline)
print("\n  [Model 1] Logistic Regression …")
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=SEED, C=0.1))
])
lr_scores = cross_val_score(lr_pipe, X, y, cv=SKF, scoring="roc_auc", n_jobs=-1)
print(f"    CV AUC: {lr_scores.mean():.4f} ± {lr_scores.std():.4f}")
lr_pipe.fit(X, y)
lr_pred = lr_pipe.predict_proba(X_test)[:, 1]


  [Model 1] Logistic Regression …
    CV AUC: 0.7635 ± 0.0022


โดย train **LightGBM** ด้วยเทคนิค **Out-of-Fold (OOF) Prediction** ผ่าน 5-Fold Stratified Cross-Validation

**Hyperparameters สำคัญ:**
- `num_leaves=127` — ควบคุมความซับซ้อนของต้นไม้
- `feature_fraction=0.8`, `bagging_fraction=0.8` — ใช้ random sampling เพื่อลด overfitting
- `lambda_l1`, `lambda_l2` — L1/L2 regularization
- `early_stopping(50)` — หยุดเมื่อ validation AUC ไม่ดีขึ้นใน 50 รอบ

**OOF Strategy:** ในแต่ละ fold โมเดลจะ train บน 4 folds และ predict อีก 1 fold ที่ไม่เคยเห็น ทำให้ได้ **OOF predictions** ที่เป็น unbiased estimate ของประสิทธิภาพจริง พร้อมกันนั้นยัง average test predictions จากทุก fold เพื่อลด variance

**บทบาทใน Pipeline:** LightGBM เป็นโมเดลหลักที่ให้ AUC สูงที่สุดในการ ensemble

In [ ]:
# ── 4b. LightGBM (main model)
print("\n  [Model 2] LightGBM …")

lgb_params = {
    "objective":       "binary",
    "metric":          "auc",
    "learning_rate":   0.05,
    "num_leaves":      127,
    "max_depth":       -1,
    "min_child_samples": 50,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":    5,
    "lambda_l1":       0.1,
    "lambda_l2":       0.1,
    "random_state":    SEED,
    "verbose":         -1,
}

oof_lgb   = np.zeros(len(X))
test_lgb  = np.zeros(len(X_test))
fold_aucs = []

for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        lgb_params, dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(period=-1)]
    )

    oof_lgb[val_idx]  = model.predict(X_val)
    test_lgb         += model.predict(X_test) / SKF.n_splits
    auc = roc_auc_score(y_val, oof_lgb[val_idx])
    fold_aucs.append(auc)
    print(f"    Fold {fold+1} AUC: {auc:.4f}  (best iter: {model.best_iteration})")

lgb_oof_auc = roc_auc_score(y, oof_lgb)
print(f"    ─── OOF AUC: {lgb_oof_auc:.4f}  mean fold AUC: {np.mean(fold_aucs):.4f} ───")




  [Model 2] LightGBM …
    Fold 1 AUC: 0.7708  (best iter: 129)
    Fold 2 AUC: 0.7749  (best iter: 99)
    Fold 3 AUC: 0.7697  (best iter: 113)
    Fold 4 AUC: 0.7760  (best iter: 104)
    Fold 5 AUC: 0.7724  (best iter: 109)
    ─── OOF AUC: 0.7727  mean fold AUC: 0.7728 ───


train **Gradient Boosting Classifier** จาก scikit-learn เพื่อใช้เป็นโมเดลเปรียบเทียบอีกตัว

- `n_estimators=200`, `max_depth=5` — โมเดลขนาดกลาง
- `subsample=0.8` — stochastic gradient boosting เพื่อลด overfitting
- ใช้ 3-Fold CV (ลดเหลือ 3 เพราะโมเดลนี้ช้ากว่า LightGBM มาก)

แม้ประสิทธิภาพมักต่ำกว่า LightGBM แต่ผลลัพธ์ใช้เปรียบเทียบใน model comparison chart เพื่อแสดงข้อดีของการใช้ LightGBM

In [ ]:
# ── 4c. Gradient Boosting (sklearn, lighter for comparison)
print("\n  [Model 3] Gradient Boosting (sklearn) …")
gb_scores = cross_val_score(
    GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                               max_depth=5, subsample=0.8, random_state=SEED),
    X, y, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
    scoring="roc_auc", n_jobs=1
)
print(f"    CV AUC: {gb_scores.mean():.4f} ± {gb_scores.std():.4f}")



  [Model 3] Gradient Boosting (sklearn) …
    CV AUC: 0.7724 ± 0.0006


สร้างกราฟ bar chart เปรียบเทียบ ROC-AUC ของทั้ง 3 โมเดล พร้อม error bar แสดง standard deviation จาก cross-validation

กราฟนี้ช่วยให้เห็นภาพรวมของประสิทธิภาพแต่ละโมเดล และสนับสนุนการตัดสินใจว่าควรใช้โมเดลใดเป็นหลักใน ensemble โดยโมเดลที่ AUC สูงสุดและ std ต่ำสุดจะถือว่าดีที่สุด

ผลลัพธ์บันทึกเป็น `model_comparison.png`

In [ ]:
# ── Model comparison chart
models = ["Logistic Reg", "LightGBM (OOF)", "Grad Boosting"]
aucs   = [lr_scores.mean(), lgb_oof_auc, gb_scores.mean()]
stds   = [lr_scores.std(),  0,            gb_scores.std()]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(models, aucs, color=["#4C72B0", "#55A868", "#DD8452"],
              yerr=stds, capsize=5)
ax.set_ylim(min(aucs) - 0.02, max(aucs) + 0.02)
ax.set_title("Model Comparison — ROC-AUC", fontsize=13, fontweight="bold")
ax.set_ylabel("AUC")
for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{val:.4f}", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/model_comparison.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✓ Saved model_comparison.png")


  ✓ Saved model_comparison.png


#  5. MODEL DIAGNOSTICS

สร้างกราฟวินิจฉัยโมเดล LightGBM จำนวน 3 กราฟ (`model_diagnostics.png`):

1. **ROC Curve (OOF)** — แสดงความสัมพันธ์ระหว่าง True Positive Rate และ False Positive Rate พร้อม AUC score ยิ่งโค้งชันและ AUC ใกล้ 1 ยิ่งดี

2. **Prediction Distribution** — แสดงการกระจายตัวของ predicted probability แยกตามกลุ่ม clicked/not-clicked หากสองกลุ่มแยกออกจากกันชัดเจน แสดงว่าโมเดล discriminate ได้ดี

3. **AUC per CV Fold** — แสดง AUC ของแต่ละ fold พร้อมเส้น mean หาก AUC ใกล้เคียงกันทุก fold แสดงว่าโมเดลมีความ stable

**บทบาทใน Pipeline:** ใช้ตรวจสอบว่าโมเดลทำงานได้สม่ำเสมอ ไม่ overfit กับ fold ใด fold หนึ่ง

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("LightGBM — Model Diagnostics", fontsize=14, fontweight="bold")



  STEP 5: MODEL DIAGNOSTICS


Text(0.5, 0.98, 'LightGBM — Model Diagnostics')

In [ ]:
# ROC curve (OOF)
fpr, tpr, _ = roc_curve(y, oof_lgb)
axes[0].plot(fpr, tpr, color="#55A868", lw=2,
             label=f"LightGBM OOF (AUC={lgb_oof_auc:.4f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve (Out-of-Fold)")
axes[0].legend()


In [ ]:
# Prediction distribution
axes[1].hist(oof_lgb[y == 0], bins=50, alpha=0.5, color="#DD8452",
             label="Not Clicked", density=True)
axes[1].hist(oof_lgb[y == 1], bins=50, alpha=0.5, color="#4C72B0",
             label="Clicked",     density=True)
axes[1].set_title("Predicted Probability Distribution")
axes[1].set_xlabel("Predicted CTR")
axes[1].legend()

In [ ]:
# Fold AUC
axes[2].bar([f"Fold {i+1}" for i in range(5)], fold_aucs,
            color="#8172B2")
axes[2].axhline(np.mean(fold_aucs), color="red", linestyle="--", label="Mean AUC")
axes[2].set_title("AUC per CV Fold")
axes[2].set_ylabel("AUC")
axes[2].legend()
axes[2].set_ylim(min(fold_aucs) - 0.005, max(fold_aucs) + 0.005)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/model_diagnostics.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✓ Saved model_diagnostics.png")



  ✓ Saved model_diagnostics.png


#  6. FEATURE IMPORTANCE & SHAP


train **Final Model** บนข้อมูล train ทั้งหมด แล้ววิเคราะห์ความสำคัญของ features ด้วย 2 วิธี

**LightGBM Feature Importance:**
- **Gain** — วัดว่า feature นั้นช่วยลด impurity ใน tree ได้มากแค่ไหน (สะท้อน predictive power)
- **Split Count** — วัดว่า feature ถูกใช้แบ่ง node บ่อยแค่ไหน (สะท้อน usage frequency)

**SHAP (SHapley Additive exPlanations):**
```python
explainer = shap.TreeExplainer(final_model)
shap_vals = explainer.shap_values(shap_sample)
```
SHAP ให้ค่า contribution ของแต่ละ feature ต่อการทำนายแต่ละ instance แตกต่างจาก feature importance ทั่วไปที่ดูภาพรวม

- **Bar Plot** — แสดง mean |SHAP value| ของแต่ละ feature (ความสำคัญเฉลี่ย)
- **Beeswarm Plot** — แสดงทิศทาง (บวก/ลบ) และขนาดของ impact ของแต่ละ feature ต่อแต่ละ sample

**บทบาทใน Pipeline:** ขั้นตอนนี้สำคัญสำหรับการอธิบายโมเดลให้ stakeholders เข้าใจ และนำ insight ไปปรับปรุง feature engineering ในรอบถัดไป

In [ ]:
# Re-train a final LightGBM on all train data
dtrain_full = lgb.Dataset(X, label=y)
final_model = lgb.train(lgb_params, dtrain_full,
                        num_boost_round=model.best_iteration,
                        callbacks=[lgb.log_evaluation(period=-1)])


In [ ]:
# LightGBM importance
imp = pd.DataFrame({
    "feature": FEATURES,
    "importance_gain": final_model.feature_importance(importance_type="gain"),
    "importance_split": final_model.feature_importance(importance_type="split"),
})
imp = imp.sort_values("importance_gain", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
top_n = 20

In [ ]:
# Gain importance
axes[0].barh(imp["feature"].head(top_n)[::-1],
             imp["importance_gain"].head(top_n)[::-1],
             color="#4C72B0")
axes[0].set_title(f"Top {top_n} Features by Gain", fontsize=13)
axes[0].set_xlabel("Gain")

Text(0.5, 58.7222222222222, 'Gain')

In [ ]:
# Split importance
imp_split = imp.sort_values("importance_split", ascending=False)
axes[1].barh(imp_split["feature"].head(top_n)[::-1],
             imp_split["importance_split"].head(top_n)[::-1],
             color="#DD8452")
axes[1].set_title(f"Top {top_n} Features by Split Count", fontsize=13)
axes[1].set_xlabel("Split Count")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/feature_importance.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✓ Saved feature_importance.png")

  ✓ Saved feature_importance.png


In [ ]:
# SHAP values (on a sample for speed)
print("  Computing SHAP values …")
shap_sample = X.sample(min(5000, len(X)), random_state=SEED)
explainer = shap.TreeExplainer(final_model)
shap_vals = explainer.shap_values(shap_sample)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
plt.sca(axes[0])
shap.summary_plot(shap_vals, shap_sample, plot_type="bar",
                  max_display=15, show=False)
axes[0].set_title("SHAP — Mean |value| (Feature Impact)", fontsize=13)

plt.sca(axes[1])
shap.summary_plot(shap_vals, shap_sample, max_display=15, show=False)
axes[1].set_title("SHAP — Beeswarm (Direction of Impact)", fontsize=13)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/shap_analysis.png", dpi=120, bbox_inches="tight")
plt.close()
print("  ✓ Saved shap_analysis.png")

print("\n  TOP 10 FEATURES BY GAIN:")
for _, row in imp.head(10).iterrows():
    print(f"    {row['feature']:<35} gain={row['importance_gain']:>10,.0f}")


  Computing SHAP values …
  ✓ Saved shap_analysis.png

  TOP 10 FEATURES BY GAIN:
    banner_area                         gain=   331,298
    banner_pos                          gain=    85,644
    historical_user_ctr                 gain=    69,827
    te_device_model                     gain=    34,292
    banner_height                       gain=    26,583
    te_app_id                           gain=    24,042
    hour                                gain=    23,965
    creative_size                       gain=    20,167
    te_site_id                          gain=    18,858
    device_type                         gain=    16,220


#  7. ENSEMBLE & SUBMISSION



รวมผลการทำนายจากหลายโมเดลด้วยเทคนิค **Weighted Blending** แล้วสร้างไฟล์ submission

```python
final_pred = 0.85 * test_lgb + 0.15 * lr_pred
```

- น้ำหนัก 85% ให้ LightGBM (โมเดลหลักที่ AUC สูงสุด)
- น้ำหนัก 15% ให้ Logistic Regression (เป็น diversity เพื่อลด variance)

ผลลัพธ์บันทึกเป็น `submission.csv` พร้อมพิมพ์สรุปสถิติ (min, max, mean) ของค่าทำนาย เพื่อตรวจสอบว่าค่าทำนายสมเหตุสมผลและใกล้เคียงกับ train CTR

**บทบาทใน Pipeline:** Ensemble มักให้ผลดีกว่าโมเดลเดี่ยว เพราะโมเดลต่างชนิดกัน capture pattern ต่างกัน การ blend ช่วยลด variance และเพิ่ม generalizability

In [ ]:
# Blend LightGBM (primary) + Logistic Reg (secondary)
BLEND_W_LGB = 0.85
BLEND_W_LR  = 0.15

final_pred = BLEND_W_LGB * test_lgb + BLEND_W_LR * lr_pred

sub["clicked"] = final_pred
sub.to_csv(f"{OUT_DIR}/submission.csv", index=False)
print(f"  ✓ Saved submission.csv")
print(f"  Prediction stats: min={final_pred.min():.4f}  "
      f"max={final_pred.max():.4f}  mean={final_pred.mean():.4f}")
print(f"  Predicted CTR : {final_pred.mean():.4f}  "
      f"(train CTR was {y.mean():.4f})")

  ✓ Saved submission.csv
  Prediction stats: min=0.0035  max=0.9393  mean=0.1990
  Predicted CTR : 0.1990  (train CTR was 0.3069)


#  8. VOLATILITY / ANOMALY ANALYSIS


วิเคราะห์ความผันผวนของ CTR ตามเวลาและตรวจจับ anomalies สร้างกราฟ 4 แบบ (`volatility_analysis.png`):

1. **Hourly CTR with Volatility Band** — แสดง CTR รายชั่วโมงพร้อม rolling mean และ ±1 std band ช่วยระบุช่วงเวลาที่ CTR ผันผวนสูง

2. **CTR Volatility by Hour** — bar chart แสดง standard deviation ของ CTR แต่ละชั่วโมง ชั่วโมงที่ std สูงบ่งชี้ความไม่แน่นอนสูง

3. **Predicted CTR by Ad Quality Quintile** — เปรียบเทียบ predicted CTR ระหว่างกลุ่มคุณภาพโฆษณา 5 ระดับ (Q1–Q5) ช่วยยืนยันว่าโมเดลสอดคล้องกับ business logic

4. **Anomaly Detection Scatter** — ระบุ impressions ที่มี predicted CTR สูงหรือต่ำผิดปกติ (top/bottom 1%) ซึ่งอาจเป็น outliers หรือ fraud

**บทบาทใน Pipeline:** การวิเคราะห์นี้ให้ business insights เพิ่มเติมและช่วยตรวจสอบความน่าเชื่อถือของโมเดล

In [ ]:
print("\n" + "=" * 60)
print("  STEP 8: TEMPORAL / VOLATILITY / ANOMALY ANALYSIS")
print("=" * 60)



  STEP 8: TEMPORAL / VOLATILITY / ANOMALY ANALYSIS


In [ ]:
# Hourly CTR time series
train_fe["clicked_int"] = y.values
hourly = (train_fe.groupby("hour")
                  .agg(impressions=("clicked_int","count"),
                       ctr=("clicked_int","mean"))
                  .reset_index())
hourly["ctr_rolling"] = hourly["ctr"].rolling(3, center=True).mean()
hourly["ctr_std"]     = hourly["ctr"].rolling(3, center=True).std().fillna(0)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Temporal & Volatility Analysis", fontsize=14, fontweight="bold")


Text(0.5, 0.98, 'Temporal & Volatility Analysis')

In [ ]:
# CTR over hours
ax = axes[0][0]
ax.fill_between(hourly["hour"],
                hourly["ctr_rolling"] - hourly["ctr_std"],
                hourly["ctr_rolling"] + hourly["ctr_std"],
                alpha=0.2, color="steelblue", label="±1 std")
ax.plot(hourly["hour"], hourly["ctr"], marker="o", color="steelblue", label="CTR")
ax.plot(hourly["hour"], hourly["ctr_rolling"], color="red", lw=2, linestyle="--",
        label="3-hr rolling mean")
ax.set_title("Hourly CTR with Volatility Band")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("CTR")
ax.legend()
ax.grid(alpha=0.3)

In [ ]:
# CTR volatility (std) by hour
ax = axes[0][1]
ax.bar(hourly["hour"], hourly["ctr_std"], color="#C44E52")
ax.set_title("CTR Volatility by Hour (rolling std)")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Std Dev")
ax.grid(alpha=0.3)

In [ ]:
# OOF score distribution by publisher quality buckets
train_fe["oof_pred"] = oof_lgb
train_fe["quality_bucket"] = pd.qcut(train_fe["ad_quality_score"], 5,
                                      labels=["Q1","Q2","Q3","Q4","Q5"],
                                      duplicates="drop")
ax = axes[1][0]
for bucket in ["Q1","Q2","Q3","Q4","Q5"]:
    subset = train_fe.loc[train_fe["quality_bucket"]==bucket, "oof_pred"]
    if len(subset) > 0:
        ax.hist(subset, bins=30, alpha=0.5, label=bucket, density=True)
ax.set_title("Predicted CTR by Ad Quality Quintile")
ax.set_xlabel("Predicted CTR")
ax.legend()

In [ ]:
# Anomaly detection: impressions with very high / low predicted CTR
train_fe["anomaly"] = (
    (train_fe["oof_pred"] > train_fe["oof_pred"].quantile(0.99)) |
    (train_fe["oof_pred"] < train_fe["oof_pred"].quantile(0.01))
).astype(int)
n_anomaly = train_fe["anomaly"].sum()
ax = axes[1][1]
ax.scatter(train_fe["ad_quality_score"],
           train_fe["oof_pred"],
           c=train_fe["anomaly"].map({0:"#4C72B0", 1:"#C44E52"}),
           alpha=0.1, s=2)
ax.set_title(f"Anomaly Detection ({n_anomaly:,} flagged = top/bot 1%)")
ax.set_xlabel("Ad Quality Score")
ax.set_ylabel("OOF Predicted CTR")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/volatility_analysis.png", dpi=120, bbox_inches="tight")
plt.close()
print(f"  ✓ Saved volatility_analysis.png")
print(f"  {n_anomaly:,} impressions flagged as anomalous CTR (top/bottom 1%)")


  ✓ Saved volatility_analysis.png
  7,492 impressions flagged as anomalous CTR (top/bottom 1%)


สรุป temporal insights ที่ค้นพบจากการวิเคราะห์:

- **Peak CTR Hour** — ชั่วโมงที่ CTR สูงสุด เหมาะสำหรับจัดสรรงบโฆษณา
- **Lowest CTR Hour** — ชั่วโมงที่ CTR ต่ำสุด ควรลดการซื้อ inventory หรือปรับ bidding strategy
- **Highest Volatility Hour** — ชั่วโมงที่ CTR ผันผวนมากที่สุด ต้องระวังในการวาง strategy เนื่องจากมีความไม่แน่นอนสูง

ข้อมูลเหล่านี้สามารถนำไปใช้ในเชิง business เพื่อตัดสินใจในการจัดสรรโฆษณาและกำหนด bidding strategy แบบ real-time

In [ ]:
# Early warning signals
peak_ctr   = hourly.loc[hourly["ctr"].idxmax(), "hour"]
trough_ctr = hourly.loc[hourly["ctr"].idxmin(), "hour"]
print(f"\n  Early Warning Signals:")
print(f"  • Peak CTR hour    : {peak_ctr}:00  ({hourly['ctr'].max():.4f})")
print(f"  • Lowest CTR hour  : {trough_ctr}:00  ({hourly['ctr'].min():.4f})")
print(f"  • Highest volatility hour: {hourly.loc[hourly['ctr_std'].idxmax(),'hour']}:00")




  Early Warning Signals:
  • Peak CTR hour    : 21:00  (0.3761)
  • Lowest CTR hour  : 2:00  (0.2199)
  • Highest volatility hour: 22:00


#  9. FINAL SUMMARY

รายงานสรุปภาพรวมทั้ง pipeline ประกอบด้วย:

- **Dataset Overview** — ขนาดและ click rate ของ train/test set
- **Features Engineered** — รายการ feature ที่สร้างขึ้นและจำนวนรวม
- **Model Performance** — เปรียบเทียบ ROC-AUC ของทุกโมเดล
- **Top Drivers (SHAP)** — 5 features ที่มีผลต่อการทำนายมากที่สุด
- **Actionable Insights** — ข้อเสนอแนะเชิง business ที่ได้จากการวิเคราะห์
- **Files Generated** — รายการไฟล์ output ทั้งหมดที่สร้างขึ้น

Cell นี้เป็นจุดสรุปที่รวบรวม key findings จากทุกขั้นตอนของ pipeline ให้อ่านได้ในที่เดียว เหมาะสำหรับใช้นำเสนอสรุปผลการทดลองต่อ stakeholders หรือส่งรายงาน

In [ ]:
print("\n" + "=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"""
  DATASET
  ─────────────────────────────────────────
  • Train: 374,590 impressions | 30.7% click rate
  • Test : 125,410 impressions | no labels
  • Target: binary classification (clicked=1)

  FEATURES ENGINEERED ({len(FEATURES)} total)
  ─────────────────────────────────────────
  • Time: is_weekend, is_peak_hour, ts_hour, ts_minute
  • Banner: width, height, area, aspect ratio
  • User: is_high_depth, ctr × quality interaction
  • Target-encoded: publisher, ad, campaign, device model, site/app IDs
  • Publisher stats: impression count, historical CTR

  MODEL PERFORMANCE (ROC-AUC)
  ─────────────────────────────────────────
  • Logistic Regression : {lr_scores.mean():.4f} ± {lr_scores.std():.4f}
  • Gradient Boosting   : {gb_scores.mean():.4f} ± {gb_scores.std():.4f}
  • LightGBM (OOF 5-cv) : {lgb_oof_auc:.4f}  ← PRIMARY MODEL
  • Ensemble (85% LGB + 15% LR): used for submission

  TOP DRIVERS (from SHAP)
  ─────────────────────────────────────────
  1. historical_user_ctr — most predictive single feature
  2. te_publisher_id     — publisher-level CTR signal
  3. ad_quality_score    — ad creative quality
  4. te_ad_id            — individual ad effectiveness
  5. user_depth          — engagement depth of user

  ACTIONABLE INSIGHTS
  ─────────────────────────────────────────
   • Focus ad delivery during peak hours (higher CTR)
  • Prioritize high-quality ads (ad_quality_score strong signal)
  • Users with high historical CTR are most likely to click again
  • Publisher selection is crucial — large spread in pub CTR
  • Larger banner sizes (300x600) outperform smaller ones

  FILES GENERATED
  ─────────────────────────────────────────
  • submission.csv          — predictions for test set
  • eda_overview.png        — class balance, CTR by day/hour
  • eda_categorical.png     — CTR by categorical features
  • eda_numerics.png        — numeric feature distributions
  • eda_correlation.png     — correlation matrix
  • model_comparison.png    — model AUC comparison
  • model_diagnostics.png   — ROC curve, prediction dist, fold AUCs
  • feature_importance.png  — top features by gain & split
  • shap_analysis.png       — SHAP beeswarm & bar plots
  • volatility_analysis.png — temporal & anomaly analysis
""")
print("  ✓ Pipeline complete!")


  FINAL SUMMARY

  DATASET
  ─────────────────────────────────────────
  • Train: 374,590 impressions | 30.7% click rate
  • Test : 125,410 impressions | no labels
  • Target: binary classification (clicked=1)

  FEATURES ENGINEERED (37 total)
  ─────────────────────────────────────────
  • Time: is_weekend, is_peak_hour, ts_hour, ts_minute
  • Banner: width, height, area, aspect ratio
  • User: is_high_depth, ctr × quality interaction
  • Target-encoded: publisher, ad, campaign, device model, site/app IDs
  • Publisher stats: impression count, historical CTR

  MODEL PERFORMANCE (ROC-AUC)
  ─────────────────────────────────────────
  • Logistic Regression : 0.7635 ± 0.0022
  • Gradient Boosting   : 0.7724 ± 0.0006
  • LightGBM (OOF 5-cv) : 0.7727  ← PRIMARY MODEL
  • Ensemble (85% LGB + 15% LR): used for submission

  TOP DRIVERS (from SHAP)
  ─────────────────────────────────────────
  1. historical_user_ctr — most predictive single feature
  2. te_publisher_id     — publisher-level